In [1]:
## Core scverse libraries
import os
import pandas as pd
import numpy as np
import scanpy as sc
import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend to prevent memory leaks
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.patches import Circle, Polygon
from scipy.spatial import cKDTree
import gc
import psutil
from concurrent.futures import ProcessPoolExecutor, as_completed

# === INPUT PATHS ===
rna_input_file = '/home/ajarrah/PhD_Thesis/chapter_3/aggregated_h5ad_data/aggregated_mouse_brain_202502.h5ad'
rna_input_folder = "/home/ajarrah/PhD_Thesis/chapter_3/h5ad_data/"
rna_tissue_positions_csv_input_path = "/home/ajarrah/PhD_Thesis/chapter_4/tissue_positions/tissue_positions.csv"
gene_image_output_dir = "/home/ajarrah/PhD_Thesis/chapter_4/images/genes/"
rna_spot_spacing = 100  # µm center-to-center
pixel_size_um = 5.0  # µm per pixel


# === Load aggregated data ===
print("Loading aggregated data...")
rna_adata = sc.read_h5ad(rna_input_file)

# === Load individual samples ===
print("Loading individual samples...")
sample_files = [
    "A1_Young_Control_Mouse_Brain_202502.h5ad",
    "B1_Young_Control_Mouse_Brain_202502.h5ad",
    "C1_Young_Control_Mouse_Brain_202502.h5ad",
    "D1_Young_Control_Mouse_Brain_202502.h5ad",
    "A1_Young_AD_Mouse_Brain_202502.h5ad",
    "B1_Young_AD_Mouse_Brain_202502.h5ad",
    "C1_Young_AD_Mouse_Brain_202502.h5ad",
    "D1_Young_AD_Mouse_Brain_202502.h5ad",
    "A1_Aged_Control_Mouse_Brain_202502.h5ad",
    "B1_Aged_Control_Mouse_Brain_202502.h5ad",
    "C1_Aged_Control_Mouse_Brain_202502.h5ad",
    "D1_Aged_Control_Mouse_Brain_202502.h5ad",
    "A1_Aged_AD_Mouse_Brain_202502.h5ad",
    "B1_Aged_AD_Mouse_Brain_202502.h5ad",
    "C1_Aged_AD_Mouse_Brain_202502.h5ad",
    "D1_Aged_AD_Mouse_Brain_202502.h5ad" 
]

rna_data_name = [sc.read_h5ad(os.path.join(rna_input_folder, f)) for f in sample_files]

rna_sample_name = [
    "YC_1", "YC_2", "YC_3", "YC_4",
    "YAD_1", "YAD_2", "YAD_3", "YAD_4",
    "AC_1", "AC_2", "AC_3", "AC_4",
    "AAD_1", "AAD_2", "AAD_3", "AAD_4"
]

rna_sample_display_name = [
    "Young Control 1", "Young Control 2", "Young Control 3", "Young Control 4",
    "Young AD 1", "Young AD 2", "Young AD 3", "Young AD 4",
    "Aged Control 1", "Aged Control 2", "Aged Control 3", "Aged Control 4",
    "Aged AD 1", "Aged AD 2", "Aged AD 3", "Aged AD 4"
]

# === Filter non-zero genes ===
print("Filtering non-zero genes...")
if isinstance(rna_adata.X, np.ndarray):
    non_zero_genes = rna_adata.X.sum(axis=0) != 0
else:
    non_zero_genes = np.array((rna_adata.X.sum(axis=0) != 0)).ravel()

rna_adata = rna_adata[:, non_zero_genes].copy()

# === Normalize data ===
print("Normalizing data...")
sc.pp.normalize_total(rna_adata, target_sum=1e2)

for data in rna_data_name:
    sc.pp.normalize_total(data, target_sum=1e2)

# === Load tissue positions ===
print("Loading tissue positions...")
tissue_positions = pd.read_csv(
    rna_tissue_positions_csv_input_path,
    usecols=["barcode", "array_row", "array_col"]
)

print(tissue_positions.head())

# === Calculate spatial coordinates ===
print("Calculating spatial coordinates...")
coords_um = []
for _, row in tissue_positions.iterrows():
    r, c = row['array_row'], row['array_col']
    y = r * (np.sqrt(3)/2 * rna_spot_spacing)
    x = c * rna_spot_spacing/2
    coords_um.append((x, y))

coords_um = np.array(coords_um)
tissue_positions['x_um'] = coords_um[:, 0]
tissue_positions['y_um'] = coords_um[:, 1]

# Clean up
del coords_um
gc.collect()

# === Filter lowly expressed genes ===
print("Filtering lowly expressed genes...")
all_genes = rna_data_name[0].var_names
gene_low_counts = {gene: 0 for gene in all_genes}

for adata in rna_data_name:
    spot_counts = np.array((adata.X > 0).sum(axis=0)).ravel()
    low_expr_mask = spot_counts <= 500
    low_genes = adata.var_names[low_expr_mask]
    for gene in low_genes:
        gene_low_counts[gene] += 1

low_expr_genes_across_15 = [gene for gene, count in gene_low_counts.items() if count >= 12]
print(f"Number of genes expressed in ≤500 spots in ≥12 samples: {len(low_expr_genes_across_15)}")

# Filter out lowly expressed genes
for i, adata in enumerate(rna_data_name):
    genes_to_keep = ~adata.var_names.isin(low_expr_genes_across_15)
    filtered = adata[:, genes_to_keep].copy()
    rna_data_name[i] = filtered
    print(f"{rna_sample_name[i]}: filtered -> {filtered.n_vars} genes remain")

# Clean up
del gene_low_counts, low_expr_genes_across_15
gc.collect()

# === Define custom colormap ===
custom_cmap = LinearSegmentedColormap.from_list("custom_scale", [
    (0.0, "#454545"),
    (0.00000001, "#000000"),
    (0.10, "#000080"),
    (0.15, "#0000FF"),
    (0.30, "#8000FF"),
    (0.45, "#FF0000"),
    (0.60, "#FF8000"),
    (0.75, "#FFFF00"),
    (1.0, "#FFFFFF")
])


def rna_plot_producer_interpolater(
    adata,
    tissue_positions,
    gene,
    rna_sample_name="YC_1",
    output_dir="plots",
    spot_diameter=55,
    cmap_name="viridis",
    pixel_size_um=5.0,
    alpha_rhombi=0.7,
    draw_poly_edges=False,
    draw_spot_edges=True,
    background_color="#00BF00"
):
    """
    Memory-safe plot generator for spatial transcriptomics with exact pixel size.
    
    Parameters:
    -----------
    pixel_size_um : float
        Target size of each pixel in micrometers (default: 5.0)
    """
    fig = None
    ax = None
    tree = None
    
    try:
        # Extract data
        spot_radius = spot_diameter / 2
        x = tissue_positions["x_um"].values.copy()
        y = tissue_positions["y_um"].values.copy()
        
        expr = adata[:, adata.var_names == gene].X
        expr = np.asarray(expr.todense()).flatten() if hasattr(expr, "todense") else np.asarray(expr).flatten()
        
        barcode_to_expr = dict(zip(adata.obs_names, expr))
        expr_full = tissue_positions["barcode"].map(barcode_to_expr).to_numpy(dtype=float)
        
        vmin, vmax = np.nanmin(expr_full), np.nanmax(expr_full)
        if not np.isfinite(vmin) or not np.isfinite(vmax):
            return
        
        # Build KDTree
        coords = np.column_stack([x, y])
        tree = cKDTree(coords)
        
        # Compute rhombi
        mid_polys, mid_vals = [], []
        for i, (xi, yi, vi) in enumerate(zip(x, y, expr_full)):
            if np.isnan(vi):
                continue
            dists, idxs = tree.query([xi, yi], k=7)
            for j in idxs[1:]:
                if np.isnan(expr_full[j]):
                    continue
                xj, yj, vj = x[j], y[j], expr_full[j]
                dx, dy = xj - xi, yj - yi
                length = np.hypot(dx, dy)
                if length == 0:
                    continue
                mx, my = (xi + xj) / 2, (yi + yj) / 2
                mv = (vi + vj) / 2
                px, py = -dy / length, dx / length
                half_short = length / (2 * np.sqrt(3))
                p1 = (mx - dx / 2, my - dy / 2)
                p2 = (mx + dx / 2, my + dy / 2)
                p3 = (mx + px * half_short, my + py * half_short)
                p4 = (mx - px * half_short, my - py * half_short)
                mid_polys.append([p1, p3, p2, p4])
                mid_vals.append(mv)
        
        # === CRITICAL: Calculate exact dimensions for 5μm pixels ===
        # Add padding for spots
        x_min = x.min() - 2 * spot_radius
        x_max = x.max() + 2 * spot_radius
        y_min = y.min() - 2 * spot_radius
        y_max = y.max() + 2 * spot_radius
        
        # Physical extent in micrometers
        width_um = x_max - x_min
        height_um = y_max - y_min
        
        # Number of pixels needed (round to ensure integer pixels)
        width_pixels = int(np.round(width_um / pixel_size_um))
        height_pixels = int(np.round(height_um / pixel_size_um))
        
        # Recalculate actual extent to match exact pixel count
        width_um = width_pixels * pixel_size_um
        height_um = height_pixels * pixel_size_um
        
        # Figure size in inches (DPI will determine pixel count)
        # We'll use 1 inch = pixel_size_um micrometers for calculation
        dpi = 25400 / pixel_size_um  # 25.4 mm per inch, converted to match pixel size
        
        fig_width_inches = width_pixels / dpi
        fig_height_inches = height_pixels / dpi
        
        #print(f"  Image: {width_pixels}×{height_pixels} px | {width_um:.1f}×{height_um:.1f} μm | DPI: {dpi:.2f}")
        
        # Create figure with exact dimensions
        fig = plt.figure(figsize=(fig_width_inches, fig_height_inches), dpi=dpi)
        fig.patch.set_facecolor(background_color)
        ax = fig.add_axes([0, 0, 1, 1])  # Fill entire figure (no margins)
        ax.set_aspect("equal")

        cmap = plt.get_cmap(cmap_name)
        norm = Normalize(vmin=vmin, vmax=vmax)
        
        # Draw rhombi
        for poly, val in zip(mid_polys, mid_vals):
            ax.add_patch(Polygon(
                poly, closed=True,
                facecolor=cmap(norm(val)),
                edgecolor="k" if draw_poly_edges else "none",
                lw=0.2,
                alpha=alpha_rhombi
            ))
        
        # Draw circles
        for xi, yi, val in zip(x, y, expr_full):
            if not np.isnan(val):
                ax.add_patch(Circle(
                    (xi, yi),
                    radius=spot_radius,
                    facecolor=cmap(norm(val)),
                    edgecolor="k" if draw_spot_edges else "none",
                    lw=0.2
                ))
        
        # Set exact limits
        ax.set_xlim(x_min, x_min + width_um)
        ax.set_ylim(y_min, y_min + height_um)
        ax.invert_yaxis()
        
        # Always turn off axis for exact pixel mapping
        ax.axis("off")
        
        # Save with exact pixel dimensions
        sample_dir = os.path.join(output_dir, rna_sample_name)
        os.makedirs(sample_dir, exist_ok=True)
        filename = f"{gene}|{rna_sample_name}|{vmin:.2f}|{vmax:.2f}.png"
        
        fig.savefig(
            os.path.join(sample_dir, filename),
            dpi=dpi,
            pad_inches=0,  # CRITICAL: No padding
            transparent=False,
            facecolor=background_color
        )
        
    except Exception as e:
        print(f"⚠️ Plot skipped for {gene}: {e}")
    
    finally:
        # Explicit cleanup
        if fig is not None:
            plt.close(fig)
        plt.close('all')
        
        # Delete all local variables
        del expr, expr_full, x, y, coords, mid_polys, mid_vals
        if tree is not None:
            del tree
        if fig is not None:
            del fig
        if ax is not None:
            del ax
        
        gc.collect()


def process_gene(args):
    """Function to run in each process"""
    adata, tissue_positions, gene, sample_name, output_dir, cmap_name = args
    
    try:
        # Slice only one gene to reduce memory
        adata_gene = adata[:, [gene]].copy()
        
        rna_plot_producer_interpolater(
            adata=adata_gene,
            tissue_positions=tissue_positions,
            gene=gene,
            rna_sample_name=sample_name,
            output_dir=output_dir,
            spot_diameter=55,
            cmap_name=cmap_name,
            pixel_size_um=pixel_size_um,
            alpha_rhombi=1,
            draw_poly_edges=False,
            draw_spot_edges=False,
            background_color="#00BF00"
        )
        
        # Clean up
        del adata_gene
        gc.collect()
        
        return f"✅ {sample_name} | {gene}"
    
    except Exception as e:
        return f"⚠️ Skipped {sample_name} | {gene}: {e}"
    
    finally:
        gc.collect()


if __name__ == "__main__":
    # System Resource Detection
    num_cores = os.cpu_count()
    total_memory_gb = round(psutil.virtual_memory().total / (1024 ** 3), 2)
    print(f"🧠 Total CPU cores available: {num_cores}")
    print(f"💾 Total system memory: {total_memory_gb} GB")
    
    # User-defined Allocation
    memory_per_worker_gb = 3  # Increased to be safer
    max_workers = max(1, num_cores - 2)  # Leave 2 cores free
    safe_memory_workers = int(total_memory_gb // memory_per_worker_gb)
    max_workers = min(max_workers, safe_memory_workers)
    print(f"🚀 Using {max_workers} workers (~{memory_per_worker_gb} GB per worker)")
    
    # Gene Processing in Batches
    BATCH_SIZE = 156  # Reduced batch size for better memory management
    
    for adata, sample_name in zip(rna_data_name, rna_sample_name):
        print(f"\n🧬 Processing RNA sample: {sample_name}")
        gene_list = adata.var_names.tolist()
        
        with ProcessPoolExecutor(max_workers=max_workers) as executor:
            for batch_start in range(0, len(gene_list), BATCH_SIZE):
                gene_batch = gene_list[batch_start:batch_start + BATCH_SIZE]
                print(f"🧩 Batch {batch_start // BATCH_SIZE + 1} ({len(gene_batch)} genes)")
                
                futures = [
                    executor.submit(
                        process_gene,
                        (adata, tissue_positions, gene, sample_name, 
                         gene_image_output_dir, custom_cmap)
                    )
                    for gene in gene_batch
                ]
                
                for f in as_completed(futures):
                    print(f.result())
                
                # Clean up after each batch
                del futures
                gc.collect()
        
        # Force garbage collection between samples
        gc.collect()
    
    print("\n🎉 All genes processed for all RNA samples.")

Loading aggregated data...


/opt/anaconda3/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Loading individual samples...
Filtering non-zero genes...


/opt/anaconda3/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Normalizing data...
Loading tissue positions...
              barcode  array_row  array_col
0  ACGCCTGACACGCGCT-1          0          0
1  TACCGATCCAACACTT-1          1          1
2  ATTAAAGCGGACGAGC-1          0          2
3  GATAAGGGACGATTAG-1          1          3
4  GTGCAAATCACCAATA-1          0          4
Calculating spatial coordinates...
Filtering lowly expressed genes...
Number of genes expressed in ≤500 spots in ≥12 samples: 24357
YC_1: filtered -> 7928 genes remain
YC_2: filtered -> 7928 genes remain
YC_3: filtered -> 7928 genes remain
YC_4: filtered -> 7928 genes remain
YAD_1: filtered -> 7928 genes remain
YAD_2: filtered -> 7928 genes remain
YAD_3: filtered -> 7928 genes remain
YAD_4: filtered -> 7928 genes remain
AC_1: filtered -> 7928 genes remain
AC_2: filtered -> 7928 genes remain
AC_3: filtered -> 7928 genes remain
AC_4: filtered -> 7928 genes remain
AAD_1: filtered -> 7928 genes remain
AAD_2: filtered -> 7928 genes remain
AAD_3: filtered -> 7928 genes remain
AAD_4: fi

In [ ]:
# Core scverse libraries
import os
import pandas as pd
import numpy as np
import scanpy as sc
import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend to prevent memory leaks
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.patches import Circle, Polygon
from scipy.spatial import cKDTree
import gc
import psutil
from concurrent.futures import ProcessPoolExecutor, as_completed

# === INPUT PATHS ===
rna_input_file = '/home/ajarrah/PhD_Thesis/chapter_3/aggregated_h5ad_data/aggregated_mouse_brain_202502.h5ad'
rna_input_folder = "/home/ajarrah/PhD_Thesis/chapter_3/h5ad_data/"
rna_tissue_positions_csv_input_path = "/home/ajarrah/PhD_Thesis/chapter_4/tissue_positions/tissue_positions.csv"
gene_image_output_dir = "/home/ajarrah/PhD_Thesis/chapter_4/images/genes/"
rna_spot_spacing = 100  # µm center-to-center
pixel_size_um = 5.0  # µm per pixel


# === Load aggregated data ===
print("Loading aggregated data...")
rna_adata = sc.read_h5ad(rna_input_file)

# === Load individual samples ===
print("Loading individual samples...")
sample_files = [
    "A1_Young_Control_Mouse_Brain_202502.h5ad",
    "B1_Young_Control_Mouse_Brain_202502.h5ad",
    "C1_Young_Control_Mouse_Brain_202502.h5ad",
    "D1_Young_Control_Mouse_Brain_202502.h5ad",
    "A1_Young_AD_Mouse_Brain_202502.h5ad",
    "B1_Young_AD_Mouse_Brain_202502.h5ad",
    "C1_Young_AD_Mouse_Brain_202502.h5ad",
    "D1_Young_AD_Mouse_Brain_202502.h5ad",
    "A1_Aged_Control_Mouse_Brain_202502.h5ad",
    "B1_Aged_Control_Mouse_Brain_202502.h5ad",
    "C1_Aged_Control_Mouse_Brain_202502.h5ad",
    "D1_Aged_Control_Mouse_Brain_202502.h5ad",
    "A1_Aged_AD_Mouse_Brain_202502.h5ad",
    "B1_Aged_AD_Mouse_Brain_202502.h5ad",
    "C1_Aged_AD_Mouse_Brain_202502.h5ad",
    "D1_Aged_AD_Mouse_Brain_202502.h5ad" 
]

rna_data_name = [sc.read_h5ad(os.path.join(rna_input_folder, f)) for f in sample_files]

rna_sample_name = [
    "YC_1", "YC_2", "YC_3", "YC_4",
    "YAD_1", "YAD_2", "YAD_3", "YAD_4",
    "AC_1", "AC_2", "AC_3", "AC_4",
    "AAD_1", "AAD_2", "AAD_3", "AAD_4"
]

rna_sample_display_name = [
    "Young Control 1", "Young Control 2", "Young Control 3", "Young Control 4",
    "Young AD 1", "Young AD 2", "Young AD 3", "Young AD 4",
    "Aged Control 1", "Aged Control 2", "Aged Control 3", "Aged Control 4",
    "Aged AD 1", "Aged AD 2", "Aged AD 3", "Aged AD 4"
]

# === Filter non-zero genes ===
print("Filtering non-zero genes...")
if isinstance(rna_adata.X, np.ndarray):
    non_zero_genes = rna_adata.X.sum(axis=0) != 0
else:
    non_zero_genes = np.array((rna_adata.X.sum(axis=0) != 0)).ravel()

rna_adata = rna_adata[:, non_zero_genes].copy()

# === Normalize data ===
print("Normalizing data...")
sc.pp.normalize_total(rna_adata, target_sum=1e2)

for data in rna_data_name:
    sc.pp.normalize_total(data, target_sum=1e2)

# === Load tissue positions ===
print("Loading tissue positions...")
tissue_positions = pd.read_csv(
    rna_tissue_positions_csv_input_path,
    usecols=["barcode", "array_row", "array_col"]
)

print(tissue_positions.head())

# === Calculate spatial coordinates ===
print("Calculating spatial coordinates...")
coords_um = []
for _, row in tissue_positions.iterrows():
    r, c = row['array_row'], row['array_col']
    y = r * (np.sqrt(3)/2 * rna_spot_spacing)
    x = c * rna_spot_spacing/2
    coords_um.append((x, y))

coords_um = np.array(coords_um)
tissue_positions['x_um'] = coords_um[:, 0]
tissue_positions['y_um'] = coords_um[:, 1]

# Clean up
del coords_um
gc.collect()

# === Filter lowly expressed genes ===
print("Filtering lowly expressed genes...")
all_genes = rna_data_name[0].var_names
gene_low_counts = {gene: 0 for gene in all_genes}

for adata in rna_data_name:
    spot_counts = np.array((adata.X > 0).sum(axis=0)).ravel()
    low_expr_mask = spot_counts <= 500
    low_genes = adata.var_names[low_expr_mask]
    for gene in low_genes:
        gene_low_counts[gene] += 1

low_expr_genes_across_15 = [gene for gene, count in gene_low_counts.items() if count >= 12]
print(f"Number of genes expressed in ≤500 spots in ≥12 samples: {len(low_expr_genes_across_15)}")

# Filter out lowly expressed genes
for i, adata in enumerate(rna_data_name):
    genes_to_keep = ~adata.var_names.isin(low_expr_genes_across_15)
    filtered = adata[:, genes_to_keep].copy()
    rna_data_name[i] = filtered
    print(f"{rna_sample_name[i]}: filtered -> {filtered.n_vars} genes remain")

# Clean up
del gene_low_counts, low_expr_genes_across_15
gc.collect()

# === Define custom colormap ===
custom_cmap = LinearSegmentedColormap.from_list("custom_scale", [
    (0.0, "#454545"),
    (0.00000001, "#000000"),
    (0.10, "#000080"),
    (0.15, "#0000FF"),
    (0.30, "#8000FF"),
    (0.45, "#FF0000"),
    (0.60, "#FF8000"),
    (0.75, "#FFFF00"),
    (1.0, "#FFFFFF")
])


def rna_plot_producer_interpolater(
    adata,
    tissue_positions,
    gene,
    rna_sample_name="YC_1",
    output_dir="plots",
    spot_diameter=55,
    cmap_name="viridis",
    pixel_size_um=5.0,
    alpha_rhombi=0.7,
    draw_poly_edges=False,
    draw_spot_edges=True,
    background_color="#00BF00"
):
    """
    Memory-safe plot generator for spatial transcriptomics with exact pixel size.
    
    Parameters:
    -----------
    pixel_size_um : float
        Target size of each pixel in micrometers (default: 5.0)
    """
    fig = None
    ax = None
    tree = None
    
    try:
        # Extract data
        spot_radius = spot_diameter / 2
        x = tissue_positions["x_um"].values.copy()
        y = tissue_positions["y_um"].values.copy()
        
        expr = adata[:, adata.var_names == gene].X
        expr = np.asarray(expr.todense()).flatten() if hasattr(expr, "todense") else np.asarray(expr).flatten()
        
        barcode_to_expr = dict(zip(adata.obs_names, expr))
        expr_full = tissue_positions["barcode"].map(barcode_to_expr).to_numpy(dtype=float)
        
        vmin, vmax = np.nanmin(expr_full), np.nanmax(expr_full)
        if not np.isfinite(vmin) or not np.isfinite(vmax):
            return
        
        # Build KDTree
        coords = np.column_stack([x, y])
        tree = cKDTree(coords)
        
        # Compute rhombi
        mid_polys, mid_vals = [], []
        for i, (xi, yi, vi) in enumerate(zip(x, y, expr_full)):
            if np.isnan(vi):
                continue
            dists, idxs = tree.query([xi, yi], k=7)
            for j in idxs[1:]:
                if np.isnan(expr_full[j]):
                    continue
                xj, yj, vj = x[j], y[j], expr_full[j]
                dx, dy = xj - xi, yj - yi
                length = np.hypot(dx, dy)
                if length == 0:
                    continue
                mx, my = (xi + xj) / 2, (yi + yj) / 2
                mv = (vi + vj) / 2
                px, py = -dy / length, dx / length
                half_short = length / (2 * np.sqrt(3))
                p1 = (mx - dx / 2, my - dy / 2)
                p2 = (mx + dx / 2, my + dy / 2)
                p3 = (mx + px * half_short, my + py * half_short)
                p4 = (mx - px * half_short, my - py * half_short)
                mid_polys.append([p1, p3, p2, p4])
                mid_vals.append(mv)
        
        # === CRITICAL: Calculate exact dimensions for 5μm pixels ===
        # Add padding for spots
        x_min = x.min() - 2 * spot_radius
        x_max = x.max() + 2 * spot_radius
        y_min = y.min() - 2 * spot_radius
        y_max = y.max() + 2 * spot_radius
        
        # Physical extent in micrometers
        width_um = x_max - x_min
        height_um = y_max - y_min
        
        # Number of pixels needed (round to ensure integer pixels)
        width_pixels = int(np.round(width_um / pixel_size_um))
        height_pixels = int(np.round(height_um / pixel_size_um))
        
        # Recalculate actual extent to match exact pixel count
        width_um = width_pixels * pixel_size_um
        height_um = height_pixels * pixel_size_um
        
        # Figure size in inches (DPI will determine pixel count)
        # We'll use 1 inch = pixel_size_um micrometers for calculation
        dpi = 25400 / pixel_size_um  # 25.4 mm per inch, converted to match pixel size
        
        fig_width_inches = width_pixels / dpi
        fig_height_inches = height_pixels / dpi
        
        #print(f"  Image: {width_pixels}×{height_pixels} px | {width_um:.1f}×{height_um:.1f} μm | DPI: {dpi:.2f}")
        
        # Create figure with exact dimensions
        fig = plt.figure(figsize=(fig_width_inches, fig_height_inches), dpi=dpi)
        fig.patch.set_facecolor(background_color)
        ax = fig.add_axes([0, 0, 1, 1])  # Fill entire figure (no margins)
        ax.set_aspect("equal")

        cmap = plt.get_cmap(cmap_name)
        norm = Normalize(vmin=vmin, vmax=vmax)
        
        # Draw rhombi
        for poly, val in zip(mid_polys, mid_vals):
            ax.add_patch(Polygon(
                poly, closed=True,
                facecolor=cmap(norm(val)),
                edgecolor="k" if draw_poly_edges else "none",
                lw=0.2,
                alpha=alpha_rhombi
            ))
        
        # Draw circles
        for xi, yi, val in zip(x, y, expr_full):
            if not np.isnan(val):
                ax.add_patch(Circle(
                    (xi, yi),
                    radius=spot_radius,
                    facecolor=cmap(norm(val)),
                    edgecolor="k" if draw_spot_edges else "none",
                    lw=0.2
                ))
        
        # Set exact limits
        ax.set_xlim(x_min, x_min + width_um)
        ax.set_ylim(y_min, y_min + height_um)
        ax.invert_yaxis()
        
        # Always turn off axis for exact pixel mapping
        ax.axis("off")
        
        # Save with exact pixel dimensions
        sample_dir = os.path.join(output_dir, rna_sample_name)
        os.makedirs(sample_dir, exist_ok=True)
        filename = f"{gene}|{rna_sample_name}|{vmin:.2f}|{vmax:.2f}.png"
        
        fig.savefig(
            os.path.join(sample_dir, filename),
            dpi=dpi,
            pad_inches=0,  # CRITICAL: No padding
            transparent=False,
            facecolor=background_color
        )
        
    except Exception as e:
        print(f"⚠️ Plot skipped for {gene}: {e}")
    
    finally:
        # Explicit cleanup
        if fig is not None:
            plt.close(fig)
        plt.close('all')
        
        # Delete all local variables
        del expr, expr_full, x, y, coords, mid_polys, mid_vals
        if tree is not None:
            del tree
        if fig is not None:
            del fig
        if ax is not None:
            del ax
        
        gc.collect()


def process_gene(args):
    """Function to run in each process"""
    adata, tissue_positions, gene, sample_name, output_dir, cmap_name = args
    
    try:
        # Slice only one gene to reduce memory
        adata_gene = adata[:, [gene]].copy()
        
        rna_plot_producer_interpolater(
            adata=adata_gene,
            tissue_positions=tissue_positions,
            gene=gene,
            rna_sample_name=sample_name,
            output_dir=output_dir,
            spot_diameter=55,
            cmap_name=cmap_name,
            pixel_size_um=pixel_size_um,
            alpha_rhombi=1,
            draw_poly_edges=False,
            draw_spot_edges=False,
            background_color="#00BF00"
        )
        
        # Clean up
        del adata_gene
        gc.collect()
        
        return f"✅ {sample_name} | {gene}"
    
    except Exception as e:
        return f"⚠️ Skipped {sample_name} | {gene}: {e}"
    
    finally:
        gc.collect()


if __name__ == "__main__":
    # System Resource Detection
    num_cores = os.cpu_count()
    total_memory_gb = round(psutil.virtual_memory().total / (1024 ** 3), 2)
    print(f"🧠 Total CPU cores available: {num_cores}")
    print(f"💾 Total system memory: {total_memory_gb} GB")
    
    # User-defined Allocation
    memory_per_worker_gb = 3  # Increased to be safer
    max_workers = max(1, num_cores - 2)  # Leave 2 cores free
    safe_memory_workers = int(total_memory_gb // memory_per_worker_gb)
    max_workers = min(max_workers, safe_memory_workers)
    print(f"🚀 Using {max_workers} workers (~{memory_per_worker_gb} GB per worker)")
    
    # Gene Processing in Batches
    BATCH_SIZE = 156  # Reduced batch size for better memory management
    
    for adata, sample_name in zip(rna_data_name, rna_sample_name):
        print(f"\n🧬 Processing RNA sample: {sample_name}")
        gene_list = adata.var_names.tolist()
        if sample_name ==  "YC_1" or sample_name == "YC_2" or sample_name == "YC_3" or sample_name == "YC_4" or sample_name == "YAD_1" or sample_name == "YAD_2" or sample_name == "YAD_3" or sample_name == "YAD_4":
            continue
        else: 
            with ProcessPoolExecutor(max_workers=max_workers) as executor:
                for batch_start in range(0, len(gene_list), BATCH_SIZE):
                    gene_batch = gene_list[batch_start:batch_start + BATCH_SIZE]
                    print(f"🧩 Batch {batch_start // BATCH_SIZE + 1} ({len(gene_batch)} genes)")
                    
                    futures = [
                        executor.submit(
                            process_gene,
                            (adata, tissue_positions, gene, sample_name, 
                            gene_image_output_dir, custom_cmap)
                        )
                        for gene in gene_batch
                    ]
                    
                    for f in as_completed(futures):
                        print(f.result())
                    
                    # Clean up after each batch
                    del futures
                    gc.collect()
            
            # Force garbage collection between samples
            gc.collect()
    
    print("\n🎉 All genes processed for all RNA samples.")

Loading aggregated data...


/opt/anaconda3/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Loading individual samples...
Filtering non-zero genes...


/opt/anaconda3/lib/python3.12/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Normalizing data...
Loading tissue positions...
              barcode  array_row  array_col
0  ACGCCTGACACGCGCT-1          0          0
1  TACCGATCCAACACTT-1          1          1
2  ATTAAAGCGGACGAGC-1          0          2
3  GATAAGGGACGATTAG-1          1          3
4  GTGCAAATCACCAATA-1          0          4
Calculating spatial coordinates...
Filtering lowly expressed genes...
Number of genes expressed in ≤500 spots in ≥12 samples: 24357
YC_1: filtered -> 7928 genes remain
YC_2: filtered -> 7928 genes remain
YC_3: filtered -> 7928 genes remain
YC_4: filtered -> 7928 genes remain
YAD_1: filtered -> 7928 genes remain
YAD_2: filtered -> 7928 genes remain
YAD_3: filtered -> 7928 genes remain
YAD_4: filtered -> 7928 genes remain
AC_1: filtered -> 7928 genes remain
AC_2: filtered -> 7928 genes remain
AC_3: filtered -> 7928 genes remain
AC_4: filtered -> 7928 genes remain
AAD_1: filtered -> 7928 genes remain
AAD_2: filtered -> 7928 genes remain
AAD_3: filtered -> 7928 genes remain
AAD_4: fi

Process ForkProcess-55:
Process ForkProcess-49:
Process ForkProcess-61:
Process ForkProcess-1:
Process ForkProcess-6:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/opt/anaconda3/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/opt/anaconda3/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/opt/anaconda3/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/opt/anaconda3/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/opt/anaconda3/lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/opt/anaconda3/lib/python3.12/multiproc

KeyboardInterrupt: 